# LoRA Contriever: Parameter-Efficient Adaptation on PubMed Abstracts

Adapts Contriever using LoRA (Low-Rank Adaptation) injected into attention layers. Same contrastive objective as DAPT but backbone is frozen — only LoRA weights are trained.

**LoRA config**: rank=16, alpha=32, target modules: query + value projections

In [6]:
!pip install torch==2.1.0 --index-url https://download.pytorch.org/whl/cu118 --upgrade -q
!pip install datasets transformers peft faiss-cpu rank_bm25 numpy tqdm psutil -q
print("Done")

ERROR: Could not find a version that satisfies the requirement torch==2.1.0 (from versions: 2.2.0+cu118, 2.2.1+cu118, 2.2.2+cu118, 2.3.0+cu118, 2.3.1+cu118, 2.4.0+cu118, 2.4.1+cu118, 2.5.0+cu118, 2.5.1+cu118, 2.6.0+cu118, 2.7.0+cu118, 2.7.1+cu118)
ERROR: No matching distribution found for torch==2.1.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 75.9 MB/s eta 0:00:00
Done


In [7]:
!pip install torchao --upgrade -q
!pip install peft --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 27.0 MB/s eta 0:00:00


In [8]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm import tqdm
import faiss
import random
import json
import time
import gc
import os

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: Tesla T4


## 1. Load Training Corpus

In [14]:
print("Loading PubMed abstracts...")
dapt_corpus_raw = load_dataset("qiaojin/PubMedQA", "pqa_artificial", split="train")

dapt_texts = []
for example in tqdm(dapt_corpus_raw, desc="Extracting"):
    abstract = " ".join(example["context"]["contexts"])
    if len(abstract.strip()) > 50:
        dapt_texts.append(abstract)

print(f"Training corpus: {len(dapt_texts)} abstracts")

Loading PubMed abstracts...


Extracting: 100%|██████████| 211269/211269 [00:41<00:00, 5127.46it/s]

Training corpus: 211266 abstracts


## 2. Dataset and Loss (same as DAPT notebook)

In [15]:
class RandomCropDataset(Dataset):
    def __init__(self, texts, tokenizer, span_tokens=64, max_length=512):
        self.texts = texts
        self.tokenizer = tokenizer
        self.span_tokens = span_tokens
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        tokens = self.tokenizer.encode(text, add_special_tokens=False, max_length=self.max_length, truncation=True)
        if len(tokens) <= self.span_tokens:
            span1 = span2 = tokens
        else:
            s1 = random.randint(0, len(tokens) - self.span_tokens)
            s2 = random.randint(0, len(tokens) - self.span_tokens)
            span1 = tokens[s1:s1 + self.span_tokens]
            span2 = tokens[s2:s2 + self.span_tokens]
        return {"span1": span1, "span2": span2}

def collate_fn(batch):
    tokenizer = collate_fn.tokenizer
    def pad(spans):
        max_len = max(len(s) for s in spans)
        ids = [s + [tokenizer.pad_token_id] * (max_len - len(s)) for s in spans]
        masks = [[1]*len(s) + [0]*(max_len - len(s)) for s in spans]
        return torch.tensor(ids, dtype=torch.long), torch.tensor(masks, dtype=torch.long)
    s1 = [b["span1"] for b in batch]
    s2 = [b["span2"] for b in batch]
    ids1, masks1 = pad(s1)
    ids2, masks2 = pad(s2)
    return {"input_ids1": ids1, "attention_mask1": masks1,
            "input_ids2": ids2, "attention_mask2": masks2}

def mean_pool(token_embeddings, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    return (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

def contrastive_loss(emb1, emb2, temperature=0.05):
    emb1 = torch.nn.functional.normalize(emb1, dim=-1)
    emb2 = torch.nn.functional.normalize(emb2, dim=-1)
    sim = torch.matmul(emb1, emb2.T) / temperature
    labels = torch.arange(sim.size(0), device=sim.device)
    return (torch.nn.functional.cross_entropy(sim, labels) +
            torch.nn.functional.cross_entropy(sim.T, labels)) / 2

print("Dataset and loss defined.")

Dataset and loss defined.


## 3. Load Contriever + Apply LoRA

LoRA injects trainable low-rank matrices into the query and value projections of each attention layer. The backbone (all other weights) is frozen.

**Config:**
- `r=16`: rank of the low-rank matrices
- `lora_alpha=32`: scaling factor
- `target_modules`: query and value projections (`query`, `value`)
- Trainable parameters: ~1.2M vs 110M total (about 1%)

In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "facebook/contriever"
print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModel.from_pretrained(model_name)

# Apply LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
)

model = get_peft_model(base_model, lora_config)
model.to(device)
model.train()

# Show trainable parameter count
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

collate_fn.tokenizer = tokenizer

Loading facebook/contriever...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: facebook/contriever
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable parameters: 589,824 / 110,072,064 (0.54%)


## 4. Training

In [17]:
BATCH_SIZE = 64
LEARNING_RATE = 1e-4   # higher LR for LoRA (fewer params)
NUM_EPOCHS = 3
MAX_STEPS_PER_EPOCH = 500
TEMPERATURE = 0.05

dataset = RandomCropDataset(dapt_texts, tokenizer, span_tokens=64)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        collate_fn=collate_fn, num_workers=2, pin_memory=True)

# Only optimize trainable (LoRA) parameters
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)

os.makedirs("checkpoints", exist_ok=True)
training_log = []
best_loss = float("inf")

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    steps = 0
    start_time = time.time()

    for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", total=MAX_STEPS_PER_EPOCH):
        if steps >= MAX_STEPS_PER_EPOCH:
            break

        ids1 = batch["input_ids1"].to(device)
        masks1 = batch["attention_mask1"].to(device)
        ids2 = batch["input_ids2"].to(device)
        masks2 = batch["attention_mask2"].to(device)

        out1 = model(input_ids=ids1, attention_mask=masks1)
        out2 = model(input_ids=ids2, attention_mask=masks2)

        emb1 = mean_pool(out1.last_hidden_state, masks1)
        emb2 = mean_pool(out2.last_hidden_state, masks2)

        loss = contrastive_loss(emb1, emb2, temperature=TEMPERATURE)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()
        steps += 1

        del ids1, masks1, ids2, masks2, out1, out2, emb1, emb2, loss

    avg_loss = epoch_loss / steps
    elapsed = time.time() - start_time
    print(f"Epoch {epoch+1}: avg_loss={avg_loss:.4f}, steps={steps}, time={elapsed:.0f}s")
    training_log.append({"epoch": epoch+1, "loss": avg_loss, "steps": steps})

    # Save LoRA adapter weights (small — just the low-rank matrices)
    model.save_pretrained(f"checkpoints/contriever_lora_epoch{epoch+1}")
    tokenizer.save_pretrained(f"checkpoints/contriever_lora_epoch{epoch+1}")
    print(f"Saved checkpoint: checkpoints/contriever_lora_epoch{epoch+1}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        model.save_pretrained("checkpoints/contriever_lora_best")
        tokenizer.save_pretrained("checkpoints/contriever_lora_best")
        print(f"New best model (loss={best_loss:.4f})")

json.dump(training_log, open("lora_training_log.json", "w"), indent=2)
print("\nTraining complete.")

Epoch 1/3: 100%|██████████| 500/500 [08:19<00:00,  1.00it/s]


Epoch 1: avg_loss=0.5583, steps=500, time=500s
Saved checkpoint: checkpoints/contriever_lora_epoch1
New best model (loss=0.5583)


Epoch 2/3: 100%|██████████| 500/500 [08:34<00:00,  1.03s/it]


Epoch 2: avg_loss=0.3793, steps=500, time=514s
Saved checkpoint: checkpoints/contriever_lora_epoch2
New best model (loss=0.3793)


Epoch 3/3: 100%|██████████| 500/500 [08:34<00:00,  1.03s/it]


Epoch 3: avg_loss=0.3434, steps=500, time=515s
Saved checkpoint: checkpoints/contriever_lora_epoch3
New best model (loss=0.3434)

Training complete.


## 5. Evaluate LoRA Contriever

In [18]:
def encode_texts(texts, tokenizer, model, batch_size=64, desc="Encoding"):
    model.eval()
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        emb = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])
        all_embeddings.append(emb.cpu().numpy())
        del inputs, outputs, emb
    return np.vstack(all_embeddings)

def evaluate_retrieval(retrieved_ids, gold_ids, k_values=[5, 20]):
    recalls = {k: 0.0 for k in k_values}
    mrr_sum = 0.0
    n = len(gold_ids)
    for i in range(n):
        gold = gold_ids[i]
        retrieved = retrieved_ids[i]
        for rank, doc_id in enumerate(retrieved):
            if doc_id == gold:
                mrr_sum += 1.0 / (rank + 1)
                break
        for k in k_values:
            if gold in retrieved[:k]:
                recalls[k] += 1.0
    results = {f"Recall@{k}": recalls[k] / n for k in k_values}
    results["MRR"] = mrr_sum / n
    return results

In [19]:
# Load evaluation data
print("Loading evaluation data...")
labeled = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
distractors_raw = load_dataset("qiaojin/PubMedQA", "pqa_artificial", split="train")

NUM_DISTRACTORS = 10000
queries = [ex["question"] for ex in labeled]
gold_abstracts = [" ".join(ex["context"]["contexts"]) for ex in labeled]
gold_passage_ids = list(range(len(gold_abstracts)))

corpus = list(gold_abstracts)
for i, ex in enumerate(distractors_raw):
    if i >= NUM_DISTRACTORS:
        break
    corpus.append(" ".join(ex["context"]["contexts"]))

print(f"Queries: {len(queries)}, Corpus: {len(corpus)}")

Loading evaluation data...


pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Queries: 1000, Corpus: 11000


In [20]:
# Load best LoRA checkpoint
from peft import PeftModel

print("Loading LoRA model...")
lora_tokenizer = AutoTokenizer.from_pretrained("checkpoints/contriever_lora_best")
lora_base = AutoModel.from_pretrained("facebook/contriever")
lora_model = PeftModel.from_pretrained(lora_base, "checkpoints/contriever_lora_best")
lora_model.to(device)

# Encode
print("Encoding corpus...")
corpus_emb = encode_texts(corpus, lora_tokenizer, lora_model, desc="Corpus")

print("Encoding queries...")
query_emb = encode_texts(queries, lora_tokenizer, lora_model, desc="Queries")

del lora_model, lora_base
gc.collect()
torch.cuda.empty_cache()

# FAISS
faiss.normalize_L2(corpus_emb)
faiss.normalize_L2(query_emb)
index = faiss.IndexFlatIP(corpus_emb.shape[1])
index.add(corpus_emb)
_, retrieved = index.search(query_emb, 20)
lora_retrieved = [r.tolist() for r in retrieved]

lora_results = evaluate_retrieval(lora_retrieved, gold_passage_ids)
print("\nLoRA Contriever Results:")
for metric, value in lora_results.items():
    print(f"  {metric}: {value:.4f}")

json.dump(lora_results, open("lora_results.json", "w"), indent=2)
np.save("corpus_emb_lora.npy", corpus_emb)
np.save("query_emb_lora.npy", query_emb)

Loading LoRA model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: facebook/contriever
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding corpus...


Corpus: 100%|██████████| 172/172 [06:38<00:00,  2.32s/it]


Encoding queries...


Queries: 100%|██████████| 16/16 [00:02<00:00,  5.51it/s]



LoRA Contriever Results:
  Recall@5: 0.9470
  Recall@20: 0.9810
  MRR: 0.8798


## 6. Compare All Retrievers

In [18]:
# Load all results from Google Drive
DRIVE_DIR = "/content/drive/MyDrive/cs505_project"

try:
    baseline = json.load(open(f"{DRIVE_DIR}/baseline_results_expanded.json"))
    bm25_results = baseline["bm25"]
    contriever_results = baseline["contriever"]
    dapt_results = json.load(open(f"{DRIVE_DIR}/dapt_results.json"))
    lora_results = json.load(open(f"{DRIVE_DIR}/lora_results.json"))
    has_baselines = True
except Exception as e:
    has_baselines = False
    print(f"Could not load results: {e}")

if has_baselines:
    print(f"\n{'Retriever':<25} {'Recall@5':<12} {'Recall@20':<12} {'MRR':<12}")
    print("-" * 61)
    print(f"{'BM25':<25} {bm25_results['Recall@5']:<12.3f} {bm25_results['Recall@20']:<12.3f} {bm25_results['MRR']:<12.3f}")
    print(f"{'Contriever (general)':<25} {contriever_results['Recall@5']:<12.3f} {contriever_results['Recall@20']:<12.3f} {contriever_results['MRR']:<12.3f}")
    print(f"{'Contriever (DAPT)':<25} {dapt_results['Recall@5']:<12.3f} {dapt_results['Recall@20']:<12.3f} {dapt_results['MRR']:<12.3f}")
    print(f"{'Contriever (LoRA)':<25} {lora_results['Recall@5']:<12.3f} {lora_results['Recall@20']:<12.3f} {lora_results['MRR']:<12.3f}")

    print("\n\nLaTeX table rows:")
    print(f"BM25 & {bm25_results['Recall@5']:.3f} & {bm25_results['Recall@20']:.3f} & {bm25_results['MRR']:.3f} \\\\")
    print(f"Contriever & {contriever_results['Recall@5']:.3f} & {contriever_results['Recall@20']:.3f} & {contriever_results['MRR']:.3f} \\\\")
    print(f"Contriever + DAPT & {dapt_results['Recall@5']:.3f} & {dapt_results['Recall@20']:.3f} & {dapt_results['MRR']:.3f} \\\\")
    print(f"Contriever + LoRA & {lora_results['Recall@5']:.3f} & {lora_results['Recall@20']:.3f} & {lora_results['MRR']:.3f} \\\\")


Retriever                 Recall@5     Recall@20    MRR         
-------------------------------------------------------------
BM25                      0.924        0.953        0.876       
Contriever (general)      0.793        0.942        0.697       
Contriever (DAPT)         0.949        0.982        0.893       
Contriever (LoRA)         0.947        0.981        0.880       


LaTeX table rows:
BM25 & 0.924 & 0.953 & 0.876 \\
Contriever & 0.793 & 0.942 & 0.697 \\
Contriever + DAPT & 0.949 & 0.982 & 0.893 \\
Contriever + LoRA & 0.947 & 0.981 & 0.880 \\


In [9]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
save_dir = "/content/drive/MyDrive/cs505_project"
os.makedirs(save_dir, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 7. Save to Google Drive

In [23]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
save_dir = "/content/drive/MyDrive/cs505_project"
os.makedirs(save_dir, exist_ok=True)

shutil.copytree("checkpoints/contriever_lora_best", f"{save_dir}/contriever_lora_best", dirs_exist_ok=True)
shutil.copy("lora_results.json", save_dir)
shutil.copy("lora_training_log.json", save_dir)
np.save(f"{save_dir}/corpus_emb_lora.npy", corpus_emb)
np.save(f"{save_dir}/query_emb_lora.npy", query_emb)

print(f"Saved to {save_dir}")

Mounted at /content/drive
Saved to /content/drive/MyDrive/cs505_project
